In [3]:
import numpy as np
import pandas as pd

from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss

# Notebook runs from notebooks/
PROCESSED_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load existing processed datasets
men = pd.read_csv(PROCESSED_DIR / "men_massey.csv")
women = pd.read_csv(PROCESSED_DIR / "women_processed.csv")

# Calculate log Massey from existing consensus rankings
assert (
    men[["Team1MasseyRank", "Team2MasseyRank"]] > 0
).all().all()

men["LogMasseyRankDiff"] = (
    np.log(men["Team1MasseyRank"])
    - np.log(men["Team2MasseyRank"])
)

print("Men:", men.shape)
print("Women:", women.shape)

display(men.head(3))

Men: (1449, 12)
Women: (961, 8)


,Season,Team1,Team2,SeedDiff,AvgScoringMarginDiff,WinRateDiff,EFGDifferentialDiff,Target,Team1MasseyRank,Team2MasseyRank,MasseyRankDiff,LogMasseyRankDiff
0,2003,1411,1421,0,9.208046,0.151724,0.045943,0,250.4,251.6,-1.2,-0.004781
1,2003,1112,1436,-15,10.309113,0.237685,0.044359,1,2.6,156.8,-154.2,-4.099460
2,2003,1113,1272,3,-1.896552,-0.172414,-0.024137,1,34.8,20.8,14.0,0.514664


In [4]:
BASE_FEATURES = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff"
]

MEN_FEATURES = BASE_FEATURES + [
    "LogMasseyRankDiff"
]

WOMEN_FEATURES = BASE_FEATURES.copy()

MEN_PARAMS = {
    "max_depth": 5,
    "learning_rate": 0.01,
    "n_estimators": 300,
    "subsample": 0.7,
    "reg_lambda": 1,
    "reg_alpha": 0,
    "random_state": 42,
    "objective": "binary:logistic",
    "eval_metric": "logloss"
}

WOMEN_PARAMS = {
    "max_depth": 1,
    "learning_rate": 0.30,
    "n_estimators": 15,
    "subsample": 0.5,
    "reg_lambda": 1,
    "reg_alpha": 0,
    "random_state": 42,
    "objective": "binary:logistic",
    "eval_metric": "logloss"
}

VALIDATION_YEARS = [2017, 2018, 2019, 2021]

# Basic dataset integrity checks
for name, df, features in [
    ("Men", men, MEN_FEATURES),
    ("Women", women, WOMEN_FEATURES)
]:
    required = [
        "Season", "Team1", "Team2", "Target"
    ] + features

    assert all(col in df.columns for col in required), (
        f"{name}: missing required columns"
    )

    assert not df[required].isna().any().any(), (
        f"{name}: missing feature or target values"
    )

    assert (df["Team1"] < df["Team2"]).all(), (
        f"{name}: invalid Team1/Team2 ordering"
    )

    assert not df.duplicated(
        ["Season", "Team1", "Team2"]
    ).any(), f"{name}: duplicate matchups"

    assert df["Target"].isin([0, 1]).all()

print("Both datasets passed the integrity checks.")

Both datasets passed the integrity checks.


In [5]:
def temporal_evaluate(
    df,
    features,
    model_params,
    validation_years,
    model_name
):
    """
    Expanding-window temporal cross-validation.

    For each validation year:
      - Train on all earlier tournament seasons
      - Predict the validation tournament
      - Calculate Brier score
      - Store individual out-of-fold predictions

    No early stopping or calibration is applied.
    """

    score_records = []
    prediction_records = []

    for year in validation_years:

        # Temporal split
        train = df[df["Season"] < year].copy()
        valid = df[df["Season"] == year].copy()

        assert not train.empty
        assert not valid.empty
        assert train["Season"].max() < year

        X_train = train[features]
        y_train = train["Target"]

        X_valid = valid[features]
        y_valid = valid["Target"]

        # Fresh model for each validation year
        model = XGBClassifier(**model_params)

        model.fit(X_train, y_train)

        # Probability that Team1 wins
        probabilities = model.predict_proba(X_valid)[:, 1]

        assert len(probabilities) == len(valid)
        assert np.isfinite(probabilities).all()
        assert ((probabilities >= 0) & (probabilities <= 1)).all()

        # Brier score for this tournament
        brier = brier_score_loss(y_valid, probabilities)

        score_records.append({
            "Model": model_name,
            "ValidationSeason": year,
            "TrainingGames": len(train),
            "ValidationGames": len(valid),
            "BrierScore": brier
        })

        # Save one prediction per validation game
        fold_predictions = valid[
            ["Season", "Team1", "Team2", "Target"]
        ].copy()

        fold_predictions["PredictedProbability"] = probabilities
        fold_predictions["Model"] = model_name
        fold_predictions["TrainingThroughSeason"] = (
            train["Season"].max()
        )
        fold_predictions["NumTrees"] = model_params["n_estimators"]

        prediction_records.append(fold_predictions)

        print(
            f"{model_name} | {year} | "
            f"Train={len(train)} | "
            f"Valid={len(valid)} | "
            f"Brier={brier:.6f}"
        )

    scores_df = pd.DataFrame(score_records)

    oof_df = pd.concat(
        prediction_records,
        ignore_index=True
    )

    mean_brier = scores_df["BrierScore"].mean()

    print(
        f"\n{model_name} mean CV Brier: "
        f"{mean_brier:.6f}\n"
    )

    return scores_df, oof_df

In [6]:
# ============================================================
# MEN
# ============================================================

men_scores, men_oof = temporal_evaluate(
    df=men,
    features=MEN_FEATURES,
    model_params=MEN_PARAMS,
    validation_years=VALIDATION_YEARS,
    model_name="Men_LogMassey_XGBoost"
)

# ============================================================
# WOMEN
# ============================================================

women_scores, women_oof = temporal_evaluate(
    df=women,
    features=WOMEN_FEATURES,
    model_params=WOMEN_PARAMS,
    validation_years=VALIDATION_YEARS,
    model_name="Women_Base4_XGBoost"
)

# ============================================================
# COMPARE RESULTS
# ============================================================

combined_scores = pd.concat(
    [men_scores, women_scores],
    ignore_index=True
)

display(
    combined_scores.pivot(
        index="ValidationSeason",
        columns="Model",
        values="BrierScore"
    ).round(6)
)

print("Mean CV Brier:")
display(
    combined_scores.groupby("Model")["BrierScore"]
    .mean()
    .to_frame("MeanCVBrier")
    .round(6)
)

# ============================================================
# REPRODUCIBILITY CHECKS
# ============================================================

EXPECTED_MEN = 0.180918
EXPECTED_WOMEN = 0.146503

actual_men = men_scores["BrierScore"].mean()
actual_women = women_scores["BrierScore"].mean()

assert np.isclose(
    actual_men, EXPECTED_MEN, atol=0.000002, rtol=0
), (
    f"Men benchmark mismatch: {actual_men:.6f}"
)

assert np.isclose(
    actual_women, EXPECTED_WOMEN, atol=0.000002, rtol=0
), (
    f"Women benchmark mismatch: {actual_women:.6f}"
)

print("Both historical benchmarks reproduced successfully.")

Men_LogMassey_XGBoost | 2017 | Train=914 | Valid=67 | Brier=0.164622
Men_LogMassey_XGBoost | 2018 | Train=981 | Valid=67 | Brier=0.214595
Men_LogMassey_XGBoost | 2019 | Train=1048 | Valid=67 | Brier=0.139790
Men_LogMassey_XGBoost | 2021 | Train=1115 | Valid=66 | Brier=0.204665

Men_LogMassey_XGBoost mean CV Brier: 0.180918

Women_Base4_XGBoost | 2017 | Train=441 | Valid=63 | Brier=0.157730
Women_Base4_XGBoost | 2018 | Train=504 | Valid=63 | Brier=0.168904
Women_Base4_XGBoost | 2019 | Train=567 | Valid=63 | Brier=0.125319
Women_Base4_XGBoost | 2021 | Train=630 | Valid=63 | Brier=0.134059

Women_Base4_XGBoost mean CV Brier: 0.146503



Model,Men_LogMassey_XGBoost,Women_Base4_XGBoost
ValidationSeason,,
2017,0.164622,0.157730
2018,0.214595,0.168904
2019,0.139790,0.125319
2021,0.204665,0.134059


Mean CV Brier:


,MeanCVBrier
Model,
Men_LogMassey_XGBoost,0.180918
Women_Base4_XGBoost,0.146503


Both historical benchmarks reproduced successfully.


In [7]:
# Save per-season Brier scores
men_scores.to_csv(
    RESULTS_DIR / "men_xgboost_cv_scores.csv",
    index=False
)

women_scores.to_csv(
    RESULTS_DIR / "women_xgboost_cv_scores.csv",
    index=False
)

# Save individual historical OOF predictions
men_oof.to_csv(
    RESULTS_DIR / "men_xgboost_oof_predictions.csv",
    index=False
)

women_oof.to_csv(
    RESULTS_DIR / "women_xgboost_oof_predictions.csv",
    index=False
)

print("Saved evaluation results to:", RESULTS_DIR)

print("\nMen OOF predictions:", len(men_oof))
print("Women OOF predictions:", len(women_oof))

display(men_oof.head())

Saved evaluation results to: ../results

Men OOF predictions: 267
Women OOF predictions: 252


,Season,Team1,Team2,Target,PredictedProbability,Model,TrainingThroughSeason,NumTrees
0,2017,1243,1448,1,0.437631,Men_LogMassey_XGBoost,2016,300
1,2017,1291,1309,1,0.416581,Men_LogMassey_XGBoost,2016,300
2,2017,1300,1413,0,0.508436,Men_LogMassey_XGBoost,2016,300
3,2017,1344,1425,0,0.388073,Men_LogMassey_XGBoost,2016,300
4,2017,1112,1315,1,0.877234,Men_LogMassey_XGBoost,2016,300


In [8]:

# ============================================================
# CELL 6: NESTED TEMPORAL EARLY STOPPING
# ============================================================

def temporal_early_stopping_evaluate(
    df,
    features,
    model_params,
    validation_years,
    model_name,
    max_rounds=500
):
    """
    Select boosting rounds using an earlier tournament season.

    For each outer validation season:
      1. Reserve the latest earlier tournament as inner validation.
      2. Train an inner model using only seasons before that.
      3. Select boosting rounds minimizing inner Brier.
      4. Refit on all seasons before the outer validation year.
      5. Evaluate once on the outer validation season.

    The outer validation labels are never used for selecting
    the number of trees.
    """

    score_records = []
    prediction_records = []
    learning_curve_records = []

    available_seasons = sorted(df["Season"].unique())

    for outer_year in validation_years:

        # ----------------------------------------------------
        # 1. TEMPORAL SPLITS
        # ----------------------------------------------------

        earlier_seasons = [
            year for year in available_seasons
            if year < outer_year
        ]

        inner_year = earlier_seasons[-1]

        inner_train = df[
            df["Season"] < inner_year
        ]

        inner_valid = df[
            df["Season"] == inner_year
        ]

        outer_train = df[
            df["Season"] < outer_year
        ]

        outer_valid = df[
            df["Season"] == outer_year
        ]

        assert not inner_train.empty
        assert not inner_valid.empty
        assert not outer_train.empty
        assert not outer_valid.empty

        assert inner_train["Season"].max() < inner_year
        assert inner_year < outer_year

        # ----------------------------------------------------
        # 2. FIT INNER MODEL WITH MAXIMUM TREE BUDGET
        # ----------------------------------------------------

        inner_params = model_params.copy()
        inner_params["n_estimators"] = max_rounds

        inner_model = XGBClassifier(**inner_params)

        inner_model.fit(
            inner_train[features],
            inner_train["Target"]
        )

        # ----------------------------------------------------
        # 3. FIND BEST ROUND USING INNER VALIDATION BRIER
        # ----------------------------------------------------

        best_round = None
        best_inner_brier = float("inf")

        for n_trees in range(1, max_rounds + 1):

            inner_probs = inner_model.predict_proba(
                inner_valid[features],
                iteration_range=(0, n_trees)
            )[:, 1]

            inner_brier = brier_score_loss(
                inner_valid["Target"],
                inner_probs
            )

            learning_curve_records.append({
                "OuterSeason": outer_year,
                "InnerSeason": inner_year,
                "NumTrees": n_trees,
                "InnerValidationBrier": inner_brier
            })

            if inner_brier < best_inner_brier:
                best_inner_brier = inner_brier
                best_round = n_trees

        # ----------------------------------------------------
        # 4. REFIT USING ALL EARLIER TOURNAMENT SEASONS
        # ----------------------------------------------------

        final_params = model_params.copy()
        final_params["n_estimators"] = best_round

        final_model = XGBClassifier(**final_params)

        final_model.fit(
            outer_train[features],
            outer_train["Target"]
        )

        # ----------------------------------------------------
        # 5. EVALUATE ON OUTER VALIDATION SEASON
        # ----------------------------------------------------

        outer_probs = final_model.predict_proba(
            outer_valid[features]
        )[:, 1]

        outer_brier = brier_score_loss(
            outer_valid["Target"],
            outer_probs
        )

        score_records.append({
            "Model": model_name,
            "ValidationSeason": outer_year,
            "InnerValidationSeason": inner_year,
            "SelectedTrees": best_round,
            "InnerBrier": best_inner_brier,
            "OuterBrier": outer_brier,
            "TrainingGames": len(outer_train),
            "ValidationGames": len(outer_valid),
            "ReachedTreeCap": best_round == max_rounds
        })

        fold_predictions = outer_valid[
            ["Season", "Team1", "Team2", "Target"]
        ].copy()

        fold_predictions["PredictedProbability"] = outer_probs
        fold_predictions["Model"] = model_name
        fold_predictions["NumTrees"] = best_round
        fold_predictions["InnerValidationSeason"] = inner_year
        fold_predictions["TrainingThroughSeason"] = (
            outer_train["Season"].max()
        )

        prediction_records.append(fold_predictions)

        print(
            f"Outer {outer_year} | "
            f"Inner {inner_year} | "
            f"Selected trees = {best_round} | "
            f"Inner Brier = {best_inner_brier:.6f} | "
            f"Outer Brier = {outer_brier:.6f}"
        )

    scores_df = pd.DataFrame(score_records)

    predictions_df = pd.concat(
        prediction_records,
        ignore_index=True
    )

    curves_df = pd.DataFrame(learning_curve_records)

    print(
        "\nMean outer CV Brier:",
        round(scores_df["OuterBrier"].mean(), 6)
    )

    return scores_df, predictions_df, curves_df


In [9]:

# ============================================================
# CELL 7: TEST MEN'S LOG MASSEY EARLY STOPPING
# ============================================================

men_es_scores, men_es_oof, men_es_curves = (
    temporal_early_stopping_evaluate(
        df=men,
        features=MEN_FEATURES,
        model_params=MEN_PARAMS,
        validation_years=VALIDATION_YEARS,
        model_name="Men_LogMassey_EarlyStopping",
        max_rounds=500
    )
)

# Compare against the fixed 300-tree baseline from Cell 4
comparison = men_scores[
    ["ValidationSeason", "BrierScore"]
].rename(columns={
    "BrierScore": "Fixed300Brier"
})

comparison = comparison.merge(
    men_es_scores[
        [
            "ValidationSeason",
            "InnerValidationSeason",
            "SelectedTrees",
            "OuterBrier",
            "ReachedTreeCap"
        ]
    ],
    on="ValidationSeason",
    validate="one_to_one"
)

comparison["Improvement"] = (
    comparison["Fixed300Brier"]
    - comparison["OuterBrier"]
)

print("\nFIXED 300 TREES VS TEMPORAL EARLY STOPPING")

display(comparison.round(6))

print("\nMEAN CV COMPARISON")

print(
    "Fixed 300 trees:",
    round(comparison["Fixed300Brier"].mean(), 6)
)

print(
    "Temporal early stopping:",
    round(comparison["OuterBrier"].mean(), 6)
)

print(
    "Improvement (positive is better):",
    round(comparison["Improvement"].mean(), 6)
)

# Save early-stopping experiment
men_es_scores.to_csv(
    RESULTS_DIR / "men_early_stopping_scores.csv",
    index=False
)

men_es_oof.to_csv(
    RESULTS_DIR / "men_early_stopping_oof.csv",
    index=False
)

men_es_curves.to_csv(
    RESULTS_DIR / "men_early_stopping_inner_curves.csv",
    index=False
)

print("\nEarly-stopping experiment saved.")


Outer 2017 | Inner 2016 | Selected trees = 199 | Inner Brier = 0.191464 | Outer Brier = 0.167872
Outer 2018 | Inner 2017 | Selected trees = 497 | Inner Brier = 0.159695 | Outer Brier = 0.222580
Outer 2019 | Inner 2018 | Selected trees = 118 | Inner Brier = 0.207985 | Outer Brier = 0.157281
Outer 2021 | Inner 2019 | Selected trees = 483 | Inner Brier = 0.135573 | Outer Brier = 0.202153

Mean outer CV Brier: 0.187471

FIXED 300 TREES VS TEMPORAL EARLY STOPPING


,ValidationSeason,Fixed300Brier,InnerValidationSeason,SelectedTrees,OuterBrier,ReachedTreeCap,Improvement
0,2017,0.164622,2016,199,0.167872,False,-0.003249
1,2018,0.214595,2017,497,0.222580,False,-0.007985
2,2019,0.139790,2018,118,0.157281,False,-0.017491
3,2021,0.204665,2019,483,0.202153,False,0.002512



MEAN CV COMPARISON
Fixed 300 trees: 0.180918
Temporal early stopping: 0.187471
Improvement (positive is better): -0.006553

Early-stopping experiment saved.


In [10]:

# ============================================================
# CELL 8: PROBABILITY DIAGNOSTICS AND CLIPPING
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

CLIP_MIN = 0.02
CLIP_MAX = 0.98

def evaluate_probability_clipping(oof_df, model_name):

    df = oof_df.copy()

    p = df["PredictedProbability"].to_numpy()
    y = df["Target"].to_numpy()

    # Basic validation
    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()
    assert np.isin(y, [0, 1]).all()
    assert not df.duplicated(
        ["Season", "Team1", "Team2"]
    ).any()

    # Apply clipping without retraining
    df["ClippedProbability"] = np.clip(
        p, CLIP_MIN, CLIP_MAX
    )

    # Squared prediction errors
    df["RawSquaredError"] = (
        df["PredictedProbability"] - df["Target"]
    ) ** 2

    df["ClippedSquaredError"] = (
        df["ClippedProbability"] - df["Target"]
    ) ** 2

    df["WasClipped"] = (
        df["PredictedProbability"] != df["ClippedProbability"]
    )

    # Per-season Brier
    yearly = (
        df.groupby("Season")
        .agg(
            Games=("Target", "size"),
            RawBrier=("RawSquaredError", "mean"),
            ClippedBrier=("ClippedSquaredError", "mean"),
            ClippedGames=("WasClipped", "sum")
        )
        .reset_index()
    )

    yearly["Improvement"] = (
        yearly["RawBrier"] - yearly["ClippedBrier"]
    )

    print(f"\n{'=' * 60}")
    print(model_name)
    print(f"{'=' * 60}")

    print("\nPER-SEASON BRIER SCORES")
    display(yearly.round(6))

    # Match our current evaluation convention:
    # unweighted mean across tournament seasons
    raw_mean = yearly["RawBrier"].mean()
    clipped_mean = yearly["ClippedBrier"].mean()

    print("\nUNWEIGHTED MEAN CV BRIER")
    print(f"Original: {raw_mean:.6f}")
    print(f"Clipped:  {clipped_mean:.6f}")
    print(f"Improvement (positive is better): "
          f"{raw_mean - clipped_mean:+.6f}")

    print("\nPREDICTION EXTREMES")
    print("Total predictions:", len(df))
    print("Predictions below 0.02:", int((p < 0.02).sum()))
    print("Predictions above 0.98:", int((p > 0.98).sum()))
    print("Predictions changed:", int(df["WasClipped"].sum()))
    print("Minimum probability:", round(p.min(), 6))
    print("Maximum probability:", round(p.max(), 6))

    # Binary predicted class confidence:
    # max(p, 1-p), because p is probability Team1 wins
    df["Confidence"] = np.maximum(p, 1 - p)
    df["CorrectPrediction"] = (
        (p >= 0.5).astype(int) == y
    ).astype(int)

    confidence_bins = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

    df["ConfidenceBin"] = pd.cut(
        df["Confidence"],
        bins=confidence_bins,
        include_lowest=True
    )

    reliability = (
        df.groupby("ConfidenceBin", observed=True)
        .agg(
            Games=("Target", "size"),
            MeanConfidence=("Confidence", "mean"),
            ActualAccuracy=("CorrectPrediction", "mean"),
            MeanSquaredError=("RawSquaredError", "mean")
        )
        .reset_index()
    )

    reliability["ConfidenceMinusAccuracy"] = (
        reliability["MeanConfidence"]
        - reliability["ActualAccuracy"]
    )

    print("\nCONFIDENCE VS ACTUAL ACCURACY")
    display(reliability.round(4))

    print(
        "\nPositive ConfidenceMinusAccuracy = "
        "potential overconfidence within that bin."
    )
    print(
        "Interpret small bins cautiously; "
        "tournament outcomes are noisy."
    )

    return df, yearly, reliability


# ============================================================
# RUN FOR BOTH CURRENT FIXED-TREE MODELS
# ============================================================

men_clip_oof, men_clip_yearly, men_reliability = (
    evaluate_probability_clipping(
        men_oof,
        "Men: Original 4 + Log Massey"
    )
)

women_clip_oof, women_clip_yearly, women_reliability = (
    evaluate_probability_clipping(
        women_oof,
        "Women: Original 4"
    )
)

# ============================================================
# SAVE EXPERIMENT OUTPUTS
# ============================================================

men_clip_yearly.to_csv(
    RESULTS_DIR / "men_clipping_cv_scores.csv",
    index=False
)

women_clip_yearly.to_csv(
    RESULTS_DIR / "women_clipping_cv_scores.csv",
    index=False
)

men_reliability.to_csv(
    RESULTS_DIR / "men_probability_reliability.csv",
    index=False
)

women_reliability.to_csv(
    RESULTS_DIR / "women_probability_reliability.csv",
    index=False
)

print("\nProbability diagnostics saved successfully.")



Men: Original 4 + Log Massey

PER-SEASON BRIER SCORES


,Season,Games,RawBrier,ClippedBrier,ClippedGames,Improvement
0,2017,67,0.164622,0.164622,0,0.0
1,2018,67,0.214595,0.214595,0,0.0
2,2019,67,0.139790,0.139790,0,0.0
3,2021,66,0.204665,0.204665,0,0.0



UNWEIGHTED MEAN CV BRIER
Original: 0.180918
Clipped:  0.180918
Improvement (positive is better): +0.000000

PREDICTION EXTREMES
Total predictions: 267
Predictions below 0.02: 0
Predictions above 0.98: 0
Predictions changed: 0
Minimum probability: 0.036998
Maximum probability: 0.939872

CONFIDENCE VS ACTUAL ACCURACY


,ConfidenceBin,Games,MeanConfidence,ActualAccuracy,MeanSquaredError,ConfidenceMinusAccuracy
0,"(0.499, 0.6]",72,0.5478,0.4722,0.2510,0.0756
1,"(0.6, 0.7]",62,0.6402,0.6613,0.2261,-0.0210
2,"(0.7, 0.8]",38,0.7541,0.7368,0.1967,0.0173
3,"(0.8, 0.9]",49,0.8603,0.8571,0.1194,0.0032
4,"(0.9, 1.0]",46,0.9296,0.9348,0.0623,-0.0052



Positive ConfidenceMinusAccuracy = potential overconfidence within that bin.
Interpret small bins cautiously; tournament outcomes are noisy.

Women: Original 4

PER-SEASON BRIER SCORES


,Season,Games,RawBrier,ClippedBrier,ClippedGames,Improvement
0,2017,63,0.157730,0.157730,0,0.0
1,2018,63,0.168904,0.168904,0,0.0
2,2019,63,0.125319,0.125319,0,0.0
3,2021,63,0.134059,0.134059,0,0.0



UNWEIGHTED MEAN CV BRIER
Original: 0.146503
Clipped:  0.146503
Improvement (positive is better): +0.000000

PREDICTION EXTREMES
Total predictions: 252
Predictions below 0.02: 0
Predictions above 0.98: 0
Predictions changed: 0
Minimum probability: 0.055601
Maximum probability: 0.962686

CONFIDENCE VS ACTUAL ACCURACY


,ConfidenceBin,Games,MeanConfidence,ActualAccuracy,MeanSquaredError,ConfidenceMinusAccuracy
0,"(0.499, 0.6]",47,0.5516,0.5319,0.2511,0.0197
1,"(0.6, 0.7]",30,0.6585,0.6333,0.2352,0.0252
2,"(0.7, 0.8]",29,0.7611,0.7586,0.1955,0.0025
3,"(0.8, 0.9]",86,0.8550,0.8605,0.1201,-0.0054
4,"(0.9, 1.0]",60,0.9275,0.9667,0.0344,-0.0392



Positive ConfidenceMinusAccuracy = potential overconfidence within that bin.
Interpret small bins cautiously; tournament outcomes are noisy.

Probability diagnostics saved successfully.


In [11]:

# ============================================================
# CELL 9: GENERATE EARLIER TEMPORAL OOF PREDICTIONS
# ============================================================

MEN_EARLIER_YEARS = list(range(2010, 2017))
WOMEN_EARLIER_YEARS = list(range(2013, 2017))

print("Generating earlier men's OOF predictions...")

men_earlier_scores, men_earlier_oof = temporal_evaluate(
    df=men,
    features=MEN_FEATURES,
    model_params=MEN_PARAMS,
    validation_years=MEN_EARLIER_YEARS,
    model_name="Men_LogMassey_XGBoost"
)

print("\nGenerating earlier women's OOF predictions...")

women_earlier_scores, women_earlier_oof = temporal_evaluate(
    df=women,
    features=WOMEN_FEATURES,
    model_params=WOMEN_PARAMS,
    validation_years=WOMEN_EARLIER_YEARS,
    model_name="Women_Base4_XGBoost"
)

# ============================================================
# COMBINE EARLIER OOF WITH THE EXISTING CV PREDICTIONS
# ============================================================

men_all_oof = pd.concat(
    [men_earlier_oof, men_oof],
    ignore_index=True
).sort_values(
    ["Season", "Team1", "Team2"]
).reset_index(drop=True)

women_all_oof = pd.concat(
    [women_earlier_oof, women_oof],
    ignore_index=True
).sort_values(
    ["Season", "Team1", "Team2"]
).reset_index(drop=True)

# ============================================================
# VALIDATE DATA INTEGRITY AND TEMPORAL ORDERING
# ============================================================

for name, df in [
    ("Men", men_all_oof),
    ("Women", women_all_oof)
]:
    assert not df.duplicated(
        ["Season", "Team1", "Team2"]
    ).any(), f"{name}: duplicate OOF predictions"

    assert (
        df["TrainingThroughSeason"] < df["Season"]
    ).all(), f"{name}: temporal ordering violation"

    assert df["PredictedProbability"].between(0, 1).all()

    assert np.isfinite(
        df["PredictedProbability"]
    ).all()

    print(
        f"{name}: {len(df)} total OOF predictions, "
        f"seasons {sorted(df['Season'].unique().tolist())}"
    )

# ============================================================
# SAVE ADDITIONAL OOF DATASETS
# ============================================================

men_all_oof.to_csv(
    RESULTS_DIR / "men_extended_oof_predictions.csv",
    index=False
)

women_all_oof.to_csv(
    RESULTS_DIR / "women_extended_oof_predictions.csv",
    index=False
)

print("\nExtended historical OOF datasets saved.")


Generating earlier men's OOF predictions...
Men_LogMassey_XGBoost | 2010 | Train=448 | Valid=64 | Brier=0.201818
Men_LogMassey_XGBoost | 2011 | Train=512 | Valid=67 | Brier=0.241342
Men_LogMassey_XGBoost | 2012 | Train=579 | Valid=67 | Brier=0.199965
Men_LogMassey_XGBoost | 2013 | Train=646 | Valid=67 | Brier=0.208428
Men_LogMassey_XGBoost | 2014 | Train=713 | Valid=67 | Brier=0.225319
Men_LogMassey_XGBoost | 2015 | Train=780 | Valid=67 | Brier=0.178935
Men_LogMassey_XGBoost | 2016 | Train=847 | Valid=67 | Brier=0.193626

Men_LogMassey_XGBoost mean CV Brier: 0.207062


Generating earlier women's OOF predictions...
Women_Base4_XGBoost | 2013 | Train=189 | Valid=63 | Brier=0.152046
Women_Base4_XGBoost | 2014 | Train=252 | Valid=63 | Brier=0.145814
Women_Base4_XGBoost | 2015 | Train=315 | Valid=63 | Brier=0.128345
Women_Base4_XGBoost | 2016 | Train=378 | Valid=63 | Brier=0.199803

Women_Base4_XGBoost mean CV Brier: 0.156502

Men: 733 total OOF predictions, seasons [2010, 2011, 2012, 2013,

In [12]:

# ============================================================
# CELL 10: TEMPORAL PLATT CALIBRATION
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

CALIBRATION_YEARS = [2017, 2018, 2019, 2021]

def probabilities_to_logits(probabilities):
    """
    Convert probabilities into log-odds.

    Clip only for numerical stability, not as a
    probability-clipping experiment.
    """
    p = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6
    )

    return np.log(p / (1 - p)).reshape(-1, 1)


def temporal_platt_calibration(
    all_oof,
    evaluation_years,
    model_name,
    min_calibration_games=100
):
    """
    For each evaluation year:
      1. Train the calibrator using earlier OOF
         predictions and actual outcomes only.
      2. Calibrate current-year OOF probabilities.
      3. Compare raw and calibrated Brier.

    The base XGBoost model is NOT retrained here.
    """

    summary_records = []
    prediction_records = []

    for year in evaluation_years:

        calibration_train = all_oof[
            all_oof["Season"] < year
        ].copy()

        calibration_test = all_oof[
            all_oof["Season"] == year
        ].copy()

        assert not calibration_test.empty

        assert (
            calibration_train["Season"] < year
        ).all()

        assert not calibration_train.duplicated(
            ["Season", "Team1", "Team2"]
        ).any()

        if len(calibration_train) < min_calibration_games:
            raise ValueError(
                f"{model_name}: only "
                f"{len(calibration_train)} earlier OOF games "
                f"available for {year} calibration."
            )

        # ====================================================
        # FIT THE PLATT CALIBRATOR
        # ====================================================

        X_cal_train = probabilities_to_logits(
            calibration_train["PredictedProbability"]
        )

        y_cal_train = calibration_train["Target"]

        X_cal_test = probabilities_to_logits(
            calibration_test["PredictedProbability"]
        )

        # L2-regularized sigmoid calibration.
        # Class 1 means Team1 wins.
        calibrator = LogisticRegression(
            C=1.0,
            max_iter=1000,
            solver="lbfgs"
        )

        calibrator.fit(
            X_cal_train,
            y_cal_train
        )

        calibrated_probabilities = (
            calibrator.predict_proba(X_cal_test)[:, 1]
        )

        # ====================================================
        # EVALUATE
        # ====================================================

        actual = calibration_test["Target"].to_numpy()

        raw_probabilities = (
            calibration_test["PredictedProbability"].to_numpy()
        )

        raw_brier = brier_score_loss(
            actual,
            raw_probabilities
        )

        calibrated_brier = brier_score_loss(
            actual,
            calibrated_probabilities
        )

        summary_records.append({
            "Model": model_name,
            "ValidationSeason": year,
            "CalibrationGames": len(calibration_train),
            "ValidationGames": len(calibration_test),
            "RawBrier": raw_brier,
            "CalibratedBrier": calibrated_brier,
            "Improvement": raw_brier - calibrated_brier,
            "PlattSlope": calibrator.coef_[0, 0],
            "PlattIntercept": calibrator.intercept_[0]
        })

        current_predictions = calibration_test[
            ["Season", "Team1", "Team2", "Target"]
        ].copy()

        current_predictions["RawProbability"] = (
            raw_probabilities
        )

        current_predictions["CalibratedProbability"] = (
            calibrated_probabilities
        )

        current_predictions["Model"] = model_name

        prediction_records.append(current_predictions)

        print(
            f"{model_name} | {year} | "
            f"Calibration games={len(calibration_train)} | "
            f"Raw={raw_brier:.6f} | "
            f"Calibrated={calibrated_brier:.6f} | "
            f"Improvement={raw_brier-calibrated_brier:+.6f}"
        )

    summary_df = pd.DataFrame(summary_records)

    predictions_df = pd.concat(
        prediction_records,
        ignore_index=True
    )

    print(
        f"\n{model_name} mean raw Brier: "
        f"{summary_df['RawBrier'].mean():.6f}"
    )

    print(
        f"{model_name} mean calibrated Brier: "
        f"{summary_df['CalibratedBrier'].mean():.6f}"
    )

    print(
        f"Mean improvement: "
        f"{summary_df['Improvement'].mean():+.6f}\n"
    )

    return summary_df, predictions_df


In [13]:

# ============================================================
# CELL 11: RUN TEMPORAL PLATT CALIBRATION
# ============================================================

men_cal_scores, men_cal_predictions = (
    temporal_platt_calibration(
        all_oof=men_all_oof,
        evaluation_years=CALIBRATION_YEARS,
        model_name="Men_LogMassey_Platt"
    )
)

women_cal_scores, women_cal_predictions = (
    temporal_platt_calibration(
        all_oof=women_all_oof,
        evaluation_years=CALIBRATION_YEARS,
        model_name="Women_Base4_Platt"
    )
)

# ============================================================
# SHOW PER-SEASON RESULTS
# ============================================================

print("\nMEN'S CALIBRATION RESULTS")
display(men_cal_scores.round(6))

print("\nWOMEN'S CALIBRATION RESULTS")
display(women_cal_scores.round(6))

# ============================================================
# SUMMARIZE CV BRIER
# ============================================================

comparison = pd.DataFrame([
    {
        "Model": "Men: Log Massey",
        "RawCVBrier": men_cal_scores["RawBrier"].mean(),
        "CalibratedCVBrier": (
            men_cal_scores["CalibratedBrier"].mean()
        )
    },
    {
        "Model": "Women: Original 4",
        "RawCVBrier": women_cal_scores["RawBrier"].mean(),
        "CalibratedCVBrier": (
            women_cal_scores["CalibratedBrier"].mean()
        )
    }
])

comparison["Improvement"] = (
    comparison["RawCVBrier"]
    - comparison["CalibratedCVBrier"]
)

print("\nOVERALL CALIBRATION COMPARISON")
display(comparison.round(6))

# Verify original benchmarks
assert np.isclose(
    men_cal_scores["RawBrier"].mean(),
    0.180918,
    atol=0.000002,
    rtol=0
)

assert np.isclose(
    women_cal_scores["RawBrier"].mean(),
    0.146503,
    atol=0.000002,
    rtol=0
)

# ============================================================
# SAVE RESULTS
# ============================================================

men_cal_scores.to_csv(
    RESULTS_DIR / "men_platt_calibration_scores.csv",
    index=False
)

women_cal_scores.to_csv(
    RESULTS_DIR / "women_platt_calibration_scores.csv",
    index=False
)

men_cal_predictions.to_csv(
    RESULTS_DIR / "men_platt_calibrated_predictions.csv",
    index=False
)

women_cal_predictions.to_csv(
    RESULTS_DIR / "women_platt_calibrated_predictions.csv",
    index=False
)

print("\nTemporal Platt calibration experiments saved.")


Men_LogMassey_Platt | 2017 | Calibration games=466 | Raw=0.164622 | Calibrated=0.174916 | Improvement=-0.010294
Men_LogMassey_Platt | 2018 | Calibration games=533 | Raw=0.214595 | Calibrated=0.213146 | Improvement=+0.001449
Men_LogMassey_Platt | 2019 | Calibration games=600 | Raw=0.139790 | Calibrated=0.147171 | Improvement=-0.007381
Men_LogMassey_Platt | 2021 | Calibration games=667 | Raw=0.204665 | Calibrated=0.203153 | Improvement=+0.001513

Men_LogMassey_Platt mean raw Brier: 0.180918
Men_LogMassey_Platt mean calibrated Brier: 0.184597
Mean improvement: -0.003678

Women_Base4_Platt | 2017 | Calibration games=252 | Raw=0.157730 | Calibrated=0.154141 | Improvement=+0.003589
Women_Base4_Platt | 2018 | Calibration games=315 | Raw=0.168904 | Calibrated=0.171483 | Improvement=-0.002579
Women_Base4_Platt | 2019 | Calibration games=378 | Raw=0.125319 | Calibrated=0.125548 | Improvement=-0.000229
Women_Base4_Platt | 2021 | Calibration games=441 | Raw=0.134059 | Calibrated=0.138774 | Improve

,Model,ValidationSeason,CalibrationGames,ValidationGames,RawBrier,CalibratedBrier,Improvement,PlattSlope,PlattIntercept
0,Men_LogMassey_Platt,2017,466,67,0.164622,0.174916,-0.010294,0.736674,0.128116
1,Men_LogMassey_Platt,2018,533,67,0.214595,0.213146,0.001449,0.785006,0.080680
2,Men_LogMassey_Platt,2019,600,67,0.139790,0.147171,-0.007381,0.760601,0.062967
3,Men_LogMassey_Platt,2021,667,66,0.204665,0.203153,0.001513,0.818898,0.054380



WOMEN'S CALIBRATION RESULTS


,Model,ValidationSeason,CalibrationGames,ValidationGames,RawBrier,CalibratedBrier,Improvement,PlattSlope,PlattIntercept
0,Women_Base4_Platt,2017,252,63,0.157730,0.154141,0.003589,0.958760,-0.171018
1,Women_Base4_Platt,2018,315,63,0.168904,0.171483,-0.002579,0.963763,-0.270982
2,Women_Base4_Platt,2019,378,63,0.125319,0.125548,-0.000229,0.936212,-0.189131
3,Women_Base4_Platt,2021,441,63,0.134059,0.138774,-0.004714,0.991144,-0.210502



OVERALL CALIBRATION COMPARISON


,Model,RawCVBrier,CalibratedCVBrier,Improvement
0,Men: Log Massey,0.180918,0.184597,-0.003678
1,Women: Original 4,0.146503,0.147486,-0.000983



Temporal Platt calibration experiments saved.


In [14]:

# ============================================================
# CELL 12: XGBOOST + LOGISTIC REGRESSION ENSEMBLE
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import brier_score_loss

RESULTS_DIR = Path("../results")
VALIDATION_YEARS = [2017, 2018, 2019, 2021]

# ============================================================
# 1. LOAD SAVED OUT-OF-FOLD PREDICTIONS
# ============================================================

men_xgb = pd.read_csv(
    RESULTS_DIR / "men_xgboost_oof_predictions.csv"
)

women_xgb = pd.read_csv(
    RESULTS_DIR / "women_xgboost_oof_predictions.csv"
)

men_lr = pd.read_csv(
    RESULTS_DIR / "logreg_men_s5_oof.csv"
)

women_lr = pd.read_csv(
    RESULTS_DIR / "logreg_women_s2_oof.csv"
)

print("LOADED PREDICTION FILES")
print("Men XGBoost:", men_xgb.shape)
print("Men LR:", men_lr.shape)
print("Women XGBoost:", women_xgb.shape)
print("Women LR:", women_lr.shape)


# ============================================================
# 2. VERIFY AND ALIGN THE PREDICTIONS
# ============================================================

def align_predictions(xgb_df, lr_df, expected_rows, label):

    keys = ["Season", "Team1", "Team2"]

    required_cols = keys + [
        "Target", "PredictedProbability"
    ]

    for name, df in [
        ("XGBoost", xgb_df),
        ("Logistic Regression", lr_df)
    ]:

        missing = set(required_cols) - set(df.columns)

        assert not missing, (
            f"{label} {name}: missing columns {missing}"
        )

        assert len(df) == expected_rows, (
            f"{label} {name}: expected {expected_rows} rows, "
            f"found {len(df)}"
        )

        assert not df.duplicated(keys).any(), (
            f"{label} {name}: duplicate matchups"
        )

        assert (df["Team1"] < df["Team2"]).all(), (
            f"{label} {name}: incorrect team ordering"
        )

        assert df["Target"].isin([0, 1]).all(), (
            f"{label} {name}: invalid targets"
        )

        probabilities = df["PredictedProbability"]

        assert probabilities.notna().all()
        assert np.isfinite(probabilities).all()
        assert probabilities.between(0, 1).all()

        assert set(df["Season"].unique()) == set(
            VALIDATION_YEARS
        ), f"{label} {name}: unexpected seasons"

        if "TrainingThroughSeason" in df.columns:
            assert (
                df["TrainingThroughSeason"] < df["Season"]
            ).all(), (
                f"{label} {name}: temporal training violation"
            )

    # Rename probabilities before merging
    xgb = xgb_df[
        keys + ["Target", "PredictedProbability"]
    ].rename(columns={
        "Target": "Target_XGB",
        "PredictedProbability": "XGBProbability"
    })

    lr = lr_df[
        keys + ["Target", "PredictedProbability"]
    ].rename(columns={
        "Target": "Target_LR",
        "PredictedProbability": "LRProbability"
    })

    # Merge using actual matchup identifiers,
    # NEVER by CSV row order.
    merged = xgb.merge(
        lr,
        on=keys,
        how="outer",
        validate="one_to_one",
        indicator=True
    )

    assert (merged["_merge"] == "both").all(), (
        f"{label}: some matchups do not match "
        "between XGBoost and Logistic Regression."
    )

    assert len(merged) == expected_rows

    # Check that both files agree on actual results
    assert (
        merged["Target_XGB"] == merged["Target_LR"]
    ).all(), (
        f"{label}: target labels differ "
        "between the model files."
    )

    merged["Target"] = merged["Target_XGB"]

    merged = merged.drop(
        columns=["Target_XGB", "Target_LR", "_merge"]
    )

    # ========================================================
    # 3. CALCULATE 50/50 ENSEMBLE PROBABILITIES
    # ========================================================

    merged["EnsembleProbability"] = (
        0.5 * merged["XGBProbability"]
        + 0.5 * merged["LRProbability"]
    )

    assert merged["EnsembleProbability"].between(0, 1).all()

    merged = merged.sort_values(keys).reset_index(drop=True)

    print(f"\n{label}: {len(merged)} matchups aligned correctly.")

    return merged


men_ensemble = align_predictions(
    men_xgb,
    men_lr,
    expected_rows=267,
    label="Men"
)

women_ensemble = align_predictions(
    women_xgb,
    women_lr,
    expected_rows=252,
    label="Women"
)


# ============================================================
# 4. EVALUATE EACH MODEL AND THE ENSEMBLE
# ============================================================

def evaluate_ensemble(df, label):

    results = []

    prediction_columns = {
        "XGBoost": "XGBProbability",
        "Logistic Regression": "LRProbability",
        "50/50 Ensemble": "EnsembleProbability"
    }

    for year in VALIDATION_YEARS:

        valid = df[df["Season"] == year]

        for model_name, column in prediction_columns.items():

            score = brier_score_loss(
                valid["Target"],
                valid[column]
            )

            results.append({
                "Dataset": label,
                "Season": year,
                "Model": model_name,
                "Games": len(valid),
                "BrierScore": score
            })

    results_df = pd.DataFrame(results)

    yearly_table = results_df.pivot(
        index="Season",
        columns="Model",
        values="BrierScore"
    )

    mean_table = (
        results_df
        .groupby("Model")["BrierScore"]
        .mean()
        .sort_values()
        .to_frame("MeanCVBrier")
    )

    print(f"\n{'=' * 60}")
    print(f"{label.upper()} — PER-SEASON BRIER")
    print(f"{'=' * 60}")

    display(yearly_table.round(6))

    print(f"\n{label.upper()} — MEAN CV BRIER")
    display(mean_table.round(6))

    xgb_mean = mean_table.loc[
        "XGBoost", "MeanCVBrier"
    ]

    lr_mean = mean_table.loc[
        "Logistic Regression", "MeanCVBrier"
    ]

    ensemble_mean = mean_table.loc[
        "50/50 Ensemble", "MeanCVBrier"
    ]

    print("\nENSEMBLE COMPARISON")
    print(
        "Improvement vs XGBoost:",
        f"{xgb_mean - ensemble_mean:+.6f}"
    )
    print(
        "Improvement vs Logistic Regression:",
        f"{lr_mean - ensemble_mean:+.6f}"
    )

    return results_df, mean_table


men_ensemble_scores, men_ensemble_summary = (
    evaluate_ensemble(men_ensemble, "Men")
)

women_ensemble_scores, women_ensemble_summary = (
    evaluate_ensemble(women_ensemble, "Women")
)


# ============================================================
# 5. VERIFY PREVIOUS XGBOOST BASELINES
# ============================================================

assert np.isclose(
    men_ensemble_summary.loc["XGBoost", "MeanCVBrier"],
    0.180918,
    atol=0.000002,
    rtol=0
), "Men XGBoost benchmark does not match."

assert np.isclose(
    women_ensemble_summary.loc["XGBoost", "MeanCVBrier"],
    0.146503,
    atol=0.000002,
    rtol=0
), "Women XGBoost benchmark does not match."

print("\nBoth XGBoost benchmarks reproduced successfully.")

print(
    "Men LR Brier:",
    round(
        men_ensemble_summary.loc[
            "Logistic Regression", "MeanCVBrier"
        ], 6
    ),
    "(Teammate reported approximately 0.1880)"
)

print(
    "Women LR Brier:",
    round(
        women_ensemble_summary.loc[
            "Logistic Regression", "MeanCVBrier"
        ], 6
    ),
    "(Teammate reported approximately 0.1460)"
)


# ============================================================
# 6. SAVE ENSEMBLE RESULTS
# ============================================================

men_ensemble.to_csv(
    RESULTS_DIR / "men_lr_xgb_ensemble_oof.csv",
    index=False
)

women_ensemble.to_csv(
    RESULTS_DIR / "women_lr_xgb_ensemble_oof.csv",
    index=False
)

men_ensemble_scores.to_csv(
    RESULTS_DIR / "men_lr_xgb_ensemble_scores.csv",
    index=False
)

women_ensemble_scores.to_csv(
    RESULTS_DIR / "women_lr_xgb_ensemble_scores.csv",
    index=False
)

print("\nEnsemble experiment completed and saved.")


LOADED PREDICTION FILES
Men XGBoost: (267, 8)
Men LR: (267, 7)
Women XGBoost: (252, 8)
Women LR: (252, 7)

Men: 267 matchups aligned correctly.

Women: 252 matchups aligned correctly.

MEN — PER-SEASON BRIER


Model,50/50 Ensemble,Logistic Regression,XGBoost
Season,,,
2017,0.166151,0.171168,0.164622
2018,0.209202,0.208592,0.214595
2019,0.149933,0.163860,0.139790
2021,0.204682,0.208319,0.204665



MEN — MEAN CV BRIER


,MeanCVBrier
Model,
XGBoost,0.180918
50/50 Ensemble,0.182492
Logistic Regression,0.187985



ENSEMBLE COMPARISON
Improvement vs XGBoost: -0.001574
Improvement vs Logistic Regression: +0.005492

WOMEN — PER-SEASON BRIER


Model,50/50 Ensemble,Logistic Regression,XGBoost
Season,,,
2017,0.150623,0.146474,0.157730
2018,0.166757,0.167311,0.168904
2019,0.121376,0.120264,0.125319
2021,0.138323,0.149935,0.134059



WOMEN — MEAN CV BRIER


,MeanCVBrier
Model,
50/50 Ensemble,0.144270
Logistic Regression,0.145996
XGBoost,0.146503



ENSEMBLE COMPARISON
Improvement vs XGBoost: +0.002233
Improvement vs Logistic Regression: +0.001726

Both XGBoost benchmarks reproduced successfully.
Men LR Brier: 0.187985 (Teammate reported approximately 0.1880)
Women LR Brier: 0.145996 (Teammate reported approximately 0.1460)

Ensemble experiment completed and saved.


In [15]:

# ============================================================
# CELL 13: MEN'S LOG MASSEY — 2022–2025 HOLDOUT
# ============================================================

from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss
import pandas as pd
import numpy as np

HOLDOUT_YEARS = [2022, 2023, 2024, 2025]

# 1. Historical training / later holdout
men_train = men[men["Season"] <= 2021].copy()
men_test = men[men["Season"].isin(HOLDOUT_YEARS)].copy()

assert men_train["Season"].max() <= 2021
assert set(men_test["Season"]) == set(HOLDOUT_YEARS)
assert len(men_test) == 268
assert not men[["Season", "Team1", "Team2"]].duplicated().any()

# 2. Fit current frozen men's model
men_holdout_model = XGBClassifier(**MEN_PARAMS)

men_holdout_model.fit(
    men_train[MEN_FEATURES],
    men_train["Target"]
)

# 3. Predict all holdout games
men_holdout = men_test[
    ["Season", "Team1", "Team2", "Target"]
].copy()

men_holdout["Probability"] = (
    men_holdout_model.predict_proba(
        men_test[MEN_FEATURES]
    )[:, 1]
)

# 4. Report yearly Brier
men_yearly = (
    men_holdout.groupby("Season")
    .apply(
        lambda g: brier_score_loss(
            g["Target"],
            g["Probability"]
        ),
        include_groups=False
    )
    .rename("LogMasseyBrier")
    .reset_index()
)

# Previous tuned original-four results
previous_men = {
    2022: 0.242093,
    2023: 0.227380,
    2024: 0.208366,
    2025: 0.173720
}

men_yearly["Original4Brier"] = (
    men_yearly["Season"].map(previous_men)
)

men_yearly["Improvement"] = (
    men_yearly["Original4Brier"]
    - men_yearly["LogMasseyBrier"]
)

print("MEN — 2022–2025 HOLDOUT")
display(men_yearly.round(6))

men_log_brier = brier_score_loss(
    men_holdout["Target"],
    men_holdout["Probability"]
)

print(f"Original 4 holdout: 0.212890")
print(f"Log Massey holdout: {men_log_brier:.6f}")
print(
    f"Improvement: "
    f"{0.212889676 - men_log_brier:+.6f}"
)

# Save results
men_holdout.to_csv(
    RESULTS_DIR / "men_logmassey_holdout_predictions.csv",
    index=False
)

men_yearly.to_csv(
    RESULTS_DIR / "men_logmassey_holdout_scores.csv",
    index=False
)


MEN — 2022–2025 HOLDOUT


,Season,LogMasseyBrier,Original4Brier,Improvement
0,2022,0.233398,0.242093,0.008695
1,2023,0.241043,0.227380,-0.013663
2,2024,0.204682,0.208366,0.003684
3,2025,0.157624,0.173720,0.016096


Original 4 holdout: 0.212890
Log Massey holdout: 0.209187
Improvement: +0.003703


In [16]:

# ============================================================
# CELL 14: WOMEN'S XGBOOST — 2022–2025 HOLDOUT
# ============================================================

women_train = women[women["Season"] <= 2021].copy()
women_test = women[
    women["Season"].isin(HOLDOUT_YEARS)
].copy()

assert women_train["Season"].max() <= 2021
assert len(women_test) == 268

women_holdout_model = XGBClassifier(**WOMEN_PARAMS)

women_holdout_model.fit(
    women_train[WOMEN_FEATURES],
    women_train["Target"]
)

women_xgb_holdout = women_test[
    ["Season", "Team1", "Team2", "Target"]
].copy()

women_xgb_holdout["XGBProbability"] = (
    women_holdout_model.predict_proba(
        women_test[WOMEN_FEATURES]
    )[:, 1]
)

# Verify the previously recorded reference score
women_xgb_brier = brier_score_loss(
    women_xgb_holdout["Target"],
    women_xgb_holdout["XGBProbability"]
)

print(
    "Women's XGBoost holdout:",
    round(women_xgb_brier, 6)
)

assert np.isclose(
    women_xgb_brier,
    0.1453367763,
    atol=0.000002,
    rtol=0
), "Women's XGBoost reference did not reproduce."

women_xgb_holdout.to_csv(
    RESULTS_DIR / "women_xgboost_holdout_predictions.csv",
    index=False
)

print("Women's XGBoost holdout predictions saved.")


Women's XGBoost holdout: 0.145337
Women's XGBoost holdout predictions saved.
